### Document loaded 

In [75]:
pip install pypdf

Note: you may need to restart the kernel to use updated packages.


In [76]:
from pypdf import PdfReader

In [77]:
reader=PdfReader("D:\my_git\DocuMind-AI\data\policy document.pdf")

In [78]:
len(reader.pages) #Pages of PDF 

35

In [79]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Text splitter

In [80]:
text_splitter=RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=50)

In [81]:
from sentence_transformers import SentenceTransformer


In [82]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="sentence-transformers/all-MiniLM-L6-v2",
    local_dir="./models/all-MiniLM-L6-v2"
)

Fetching 30 files: 100%|██████████| 30/30 [00:00<00:00, 240.00it/s]


'D:\\my_git\\DocuMind-AI\\notebook\\models\\all-MiniLM-L6-v2'

### Model loaded

In [83]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("./models/all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4030.88it/s]


In [84]:
sentence = "Employees are entitled to 12 casual leaves annually."

In [85]:
embedding=model.encode(sentence)

In [86]:
type(embedding)

numpy.ndarray

In [87]:
embedding.shape

(384,)

In [88]:
embedding[:10]

array([ 0.06135637,  0.0352527 ,  0.03928021,  0.00827767,  0.0378606 ,
        0.07148534, -0.02701925, -0.0080509 , -0.08710077,  0.01895502],
      dtype=float32)

## Vector database

In [89]:
pip install faiss-cpu

Note: you may need to restart the kernel to use updated packages.


In [90]:
import faiss

In [91]:
print(faiss.__version__)

1.14.3


## Retriver


In [92]:
chunks = [
    "Employees receive 12 casual leaves annually.",
    "Employees must follow the company dress code.",
    "Office parking is available for all staff."
]
chunks_embedding=model.encode(chunks)

In [93]:
len(chunks_embedding)

3

## PDF reader

In [94]:
reader=PdfReader("D:\my_git\DocuMind-AI\data\policy document.pdf")

In [95]:
text=""
for page in reader.pages:
    text+=page.extract_text()

In [96]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [97]:
text_splitter=RecursiveCharacterTextSplitter(chunk_size=500,
    chunk_overlap=50)

In [98]:
chunks = text_splitter.split_text(text)

In [99]:
print(chunks[0])

MODEL EMPLOYEE HANDBOOK 
FOR SMALL BUSINESS
  INTRODUCTION 
 
The NFIB Legal Foundation is pleased to provide you with this Model Employee 
Handbook for Small Business. This handbook is intended to assist you in creating your 
own custom employee handbook. The actual polices and procedures of your business 
may vary due to the size of the company, the number of employees, benefits offered 
and other factors. The handbook is therefore intentionally broad, and should be


## Sentence Transformer

In [100]:
from sentence_transformers import SentenceTransformer

In [101]:
model=SentenceTransformer("models/all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3023.06it/s]


In [102]:
chunks_embedding=model.encode(chunks)

In [103]:
print(chunks_embedding.shape)

(126, 384)


In [104]:
print(chunks[0])

MODEL EMPLOYEE HANDBOOK 
FOR SMALL BUSINESS
  INTRODUCTION 
 
The NFIB Legal Foundation is pleased to provide you with this Model Employee 
Handbook for Small Business. This handbook is intended to assist you in creating your 
own custom employee handbook. The actual polices and procedures of your business 
may vary due to the size of the company, the number of employees, benefits offered 
and other factors. The handbook is therefore intentionally broad, and should be


In [105]:
print(chunks_embedding[0][0:10])

[ 0.04749426  0.05449278  0.01325135 -0.01392823  0.03232608  0.10258424
 -0.00540978  0.01380085 -0.04634259 -0.04304724]


## FAISS

In [106]:
import faiss
import numpy as np

In [107]:
embeddings=np.array(chunks_embedding).astype("float32")

In [108]:
dimension=embeddings.shape[1]
index=faiss.IndexFlatL2(dimension)

In [109]:
index.add(embeddings)

In [110]:
print(index.ntotal)

126


## Query question

In [111]:
query = "What is the leave policy?"

In [113]:
query_embedding = model.encode([query])

In [114]:
query_embedding = query_embedding.astype("float32")

In [115]:
k = 3

distances, indices = index.search(query_embedding, k)

In [ ]:
print(indices)

[[11 15  8]]


In [118]:
for idx in indices[0]:
    print("=" * 80)
    print(chunks[idx])

6.1. Vacation 
6.2. Sick Leave 
6.3. Family and Medical Leave Act 
6.4. Holidays 
6.5. Jury Duty 
6.6. Voting 
6.7. Military Leave 
6.8. Leave of Absence 
 ii 
7. Work Performance 
7.1. Expectations 
7.2. Reviews 
7.3. Insubordination 
 
8. Discipline Policy 
8.1. Grounds for Disciplinary Action 
8.2. Procedures 
8.3. Termination 
 
9. Employee Health and Safety 
9.1. Workplace Safety 
9.2. Workplace Security 
9.3. Emergency Procedures 
 
10. Benefits 
10.1. Health Insurance
an employee to accept employment with the company.  
The company reserves the right to unilaterally revise, suspend, revoke, terminate or 
change any of its policies, in whole or in part, whether described within this handbook 
or elsewhere, in its sole discretion. If any discrepancy between this handbook and 
current company policy arises, conform to current company policy. Every effort will be 
made to keep you informed of the company’s policies, however we cannot guarantee
2.2); (4) a section that describes the 